# 04 revenue data preparation

This GitHub copy has been cleaned for readability: duplicate/empty cells were removed and saved cell outputs were cleared. Code and analysis workflow are retained.

**Execution note:** Original project paths may reference Google Drive/Colab. Update paths and provide required source files/API credentials in your own environment before rerunning.

# Notebook 04 — Revenue Data Preparation

## Purpose

Prepare the final revenue dataset for the 130 selected Indian movies by integrating:

- Verified master movie data
- Verified TMDB movie mapping
- TMDB financial metadata
- India Gross data
- TMDB worldwide revenue
- TMDB budget

The output is a validated 130-movie revenue dataset for downstream analysis.

In [ ]:
# ============================================================
# SETUP
# ============================================================

import os
import sys
import pandas as pd

from google.colab import drive

# Mount Google Drive only if it is not already mounted
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

PROJECT_ROOT = "/content/drive/MyDrive/IndianCinemaTrendAnalytics"

print("=" * 60)
print("REVENUE DATA PREPARATION")
print("=" * 60)
print("Project root:", PROJECT_ROOT)

In [ ]:
# ============================================================
# INPUT / OUTPUT PATHS
# ============================================================

MASTER_PATH = os.path.join(
    PROJECT_ROOT,
    "01_data",
    "processed",
    "ICMB_130_MASTER_DATA.csv"
)

MAPPING_PATH = os.path.join(
    PROJECT_ROOT,
    "01_data",
    "processed",
    "ICMB_130_TMDB_MAPPING.csv"
)

FINANCIALS_PATH = os.path.join(
    PROJECT_ROOT,
    "01_data",
    "raw",
    "ICMB_130_TMDB_FINANCIALS.csv"
)

OUTPUT_PATH = os.path.join(
    PROJECT_ROOT,
    "01_data",
    "processed",
    "ICMB_130_REVENUE_DATA.csv"
)

print("=" * 60)
print("INPUT FILE CHECK")
print("=" * 60)

print("Master:", os.path.exists(MASTER_PATH))
print("Mapping:", os.path.exists(MAPPING_PATH))
print("Financials:", os.path.exists(FINANCIALS_PATH))

In [ ]:
# ============================================================
# VERIFIED INPUT CHECK
# ============================================================

required_files = {
    "Master": MASTER_PATH,
    "TMDB Mapping": MAPPING_PATH,
    "TMDB Financials": FINANCIALS_PATH
}

for name, path in required_files.items():
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{name} file not found:\n{path}"
        )

print("=" * 60)
print("ALL INPUT FILES VERIFIED")
print("=" * 60)

for name, path in required_files.items():
    print(f"{name}: {path}")

In [ ]:
# ============================================================
# LOAD VERIFIED INPUT DATASETS
# ============================================================

master_df = pd.read_csv(MASTER_PATH)
mapping_df = pd.read_csv(MAPPING_PATH)
financials_df = pd.read_csv(FINANCIALS_PATH)

print("=" * 60)
print("LOADING VERIFIED INPUT DATASETS")
print("=" * 60)

print("\nMASTER DATA")
print("Rows:", len(master_df))
print("Columns:", len(master_df.columns))

print("\nTMDB MAPPING")
print("Rows:", len(mapping_df))
print("Columns:", len(mapping_df.columns))

print("\nTMDB FINANCIALS")
print("Rows:", len(financials_df))
print("Columns:", len(financials_df.columns))

In [ ]:
# ============================================================
# 130-MOVIE ALIGNMENT VALIDATION
# ============================================================

print("=" * 60)
print("130-MOVIE ALIGNMENT VALIDATION")
print("=" * 60)

master_ids = set(master_df["movie_record_id"])
mapping_ids = set(mapping_df["movie_record_id"])
financial_ids = set(financials_df["movie_record_id"])

print("\nUnique movie_record_id counts:")
print("Master:", len(master_ids))
print("Mapping:", len(mapping_ids))
print("Financials:", len(financial_ids))

print("\nMaster → Mapping")
print("Missing:", master_ids - mapping_ids)
print("Extra:", mapping_ids - master_ids)

print("\nMaster → Financials")
print("Missing:", master_ids - financial_ids)
print("Extra:", financial_ids - master_ids)

print("\nDuplicate movie_record_id:")
print("Master:", master_df["movie_record_id"].duplicated().sum())
print("Mapping:", mapping_df["movie_record_id"].duplicated().sum())
print("Financials:", financials_df["movie_record_id"].duplicated().sum())

assert len(master_ids) == 130
assert len(mapping_ids) == 130
assert len(financial_ids) == 130

assert master_ids == mapping_ids
assert master_ids == financial_ids

assert master_df["movie_record_id"].duplicated().sum() == 0
assert mapping_df["movie_record_id"].duplicated().sum() == 0
assert financials_df["movie_record_id"].duplicated().sum() == 0

print("\n✅ 130-MOVIE ALIGNMENT VALIDATION PASSED")

In [ ]:
# ============================================================
# PREPARE TMDB MAPPING
# ============================================================

mapping_cols = [
    "movie_record_id",
    "tmdb_id",
    "tmdb_title"
]

mapping_clean = mapping_df[mapping_cols].copy()

mapping_clean["tmdb_id"] = pd.to_numeric(
    mapping_clean["tmdb_id"],
    errors="coerce"
).astype("Int64")

print("=" * 60)
print("TMDB MAPPING VALIDATION")
print("=" * 60)

print("Rows:", len(mapping_clean))
print("Missing TMDB IDs:",
      mapping_clean["tmdb_id"].isna().sum())
print("Missing TMDB titles:",
      mapping_clean["tmdb_title"].isna().sum())
print("Duplicate TMDB IDs:",
      mapping_clean["tmdb_id"].duplicated().sum())

assert len(mapping_clean) == 130
assert mapping_clean["tmdb_id"].isna().sum() == 0
assert mapping_clean["tmdb_title"].isna().sum() == 0
assert mapping_clean["tmdb_id"].duplicated().sum() == 0

print("\n✅ TMDB MAPPING VALIDATION PASSED")

In [ ]:
# ============================================================
# PREPARE TMDB FINANCIAL DATA
# ============================================================

financial_cols = [
    "movie_record_id",
    "tmdb_id",
    "budget",
    "revenue"
]

financial_clean = financials_df[financial_cols].copy()

financial_clean = financial_clean.rename(
    columns={
        "budget": "tmdb_budget_usd",
        "revenue": "tmdb_worldwide_revenue_usd"
    }
)

financial_clean["tmdb_id"] = pd.to_numeric(
    financial_clean["tmdb_id"],
    errors="coerce"
).astype("Int64")

financial_clean["tmdb_budget_usd"] = pd.to_numeric(
    financial_clean["tmdb_budget_usd"],
    errors="coerce"
)

financial_clean["tmdb_worldwide_revenue_usd"] = pd.to_numeric(
    financial_clean["tmdb_worldwide_revenue_usd"],
    errors="coerce"
)

print("=" * 60)
print("TMDB FINANCIAL DATA VALIDATION")
print("=" * 60)

print("Rows:", len(financial_clean))
print("Missing TMDB IDs:",
      financial_clean["tmdb_id"].isna().sum())
print("Missing budgets:",
      financial_clean["tmdb_budget_usd"].isna().sum())
print("Missing worldwide revenue:",
      financial_clean["tmdb_worldwide_revenue_usd"].isna().sum())

assert len(financial_clean) == 130
assert financial_clean["tmdb_id"].isna().sum() == 0
assert financial_clean["movie_record_id"].duplicated().sum() == 0

print("\n✅ TMDB FINANCIAL DATA VALIDATION PASSED")

In [ ]:
# ============================================================
# PREPARE MASTER REVENUE DATA
# ============================================================

master_revenue = master_df[
    [
        "movie_record_id",
        "year",
        "rank",
        "movie_title",
        "language",
        "india_gross_cr"
    ]
].copy()

master_revenue = master_revenue.rename(
    columns={
        "india_gross_cr": "india_gross_inr_cr"
    }
)

master_revenue["india_gross_inr_cr"] = pd.to_numeric(
    master_revenue["india_gross_inr_cr"],
    errors="coerce"
)

print("=" * 60)
print("INDIA GROSS VALIDATION")
print("=" * 60)

print("Records:", len(master_revenue))
print("Missing:", master_revenue["india_gross_inr_cr"].isna().sum())
print("Minimum:", master_revenue["india_gross_inr_cr"].min())
print("Maximum:", master_revenue["india_gross_inr_cr"].max())

assert len(master_revenue) == 130
assert master_revenue["india_gross_inr_cr"].isna().sum() == 0

print("\n✅ INDIA GROSS VALIDATION PASSED")

In [ ]:
# ============================================================
# FINAL REVENUE DATA MERGE
# ============================================================

print("=" * 60)
print("FINAL REVENUE DATA MERGE")
print("=" * 60)

revenue_df = master_revenue.merge(
    mapping_clean,
    on="movie_record_id",
    how="inner",
    validate="one_to_one"
)

revenue_df = revenue_df.merge(
    financial_clean[
        [
            "movie_record_id",
            "tmdb_budget_usd",
            "tmdb_worldwide_revenue_usd"
        ]
    ],
    on="movie_record_id",
    how="inner",
    validate="one_to_one"
)

print("Merge completed successfully.")
print("Rows:", len(revenue_df))
print("Columns:", len(revenue_df.columns))

In [ ]:
# ============================================================
# FINAL REVENUE DATASET VALIDATION
# ============================================================

print("=" * 60)
print("FINAL REVENUE DATASET VALIDATION")
print("=" * 60)

print("\nRows:", len(revenue_df))
print("Columns:", len(revenue_df.columns))

print("\nMissing values:")

for col in revenue_df.columns:
    print(
        f"{col}:",
        revenue_df[col].isna().sum()
    )

print("\nDuplicate movie records:",
      revenue_df["movie_record_id"].duplicated().sum())

print("Duplicate TMDB IDs:",
      revenue_df["tmdb_id"].duplicated().sum())

assert len(revenue_df) == 130
assert revenue_df["movie_record_id"].nunique() == 130
assert revenue_df["tmdb_id"].nunique() == 130

assert revenue_df["movie_record_id"].isna().sum() == 0
assert revenue_df["tmdb_id"].isna().sum() == 0
assert revenue_df["movie_title"].isna().sum() == 0
assert revenue_df["language"].isna().sum() == 0
assert revenue_df["india_gross_inr_cr"].isna().sum() == 0
assert revenue_df["tmdb_title"].isna().sum() == 0
assert revenue_df["tmdb_budget_usd"].isna().sum() == 0
assert revenue_df["tmdb_worldwide_revenue_usd"].isna().sum() == 0

assert revenue_df["movie_record_id"].duplicated().sum() == 0
assert revenue_df["tmdb_id"].duplicated().sum() == 0

print("\nFinal columns:")
print(list(revenue_df.columns))

print("\nFirst 10 records:")
display(revenue_df.head(10))

print("\n✅ FINAL REVENUE DATASET VALIDATION PASSED")

In [ ]:
# ============================================================
# REVENUE QUALITY SUMMARY
# ============================================================

print("=" * 60)
print("REVENUE QUALITY SUMMARY")
print("=" * 60)

print("\nIndia Gross:")
print("Missing:",
      revenue_df["india_gross_inr_cr"].isna().sum())
print("Minimum:",
      revenue_df["india_gross_inr_cr"].min())
print("Maximum:",
      revenue_df["india_gross_inr_cr"].max())

print("\nTMDB Worldwide Revenue:")
print("Missing:",
      revenue_df["tmdb_worldwide_revenue_usd"].isna().sum())
print("Zero values:",
      (revenue_df["tmdb_worldwide_revenue_usd"] == 0).sum())

print("\nTMDB Budget:")
print("Missing:",
      revenue_df["tmdb_budget_usd"].isna().sum())
print("Zero values:",
      (revenue_df["tmdb_budget_usd"] == 0).sum())

In [ ]:
# ============================================================
# SAVE FINAL REVENUE DATASET
# ============================================================

OUTPUT_DIR = os.path.join(
    PROJECT_ROOT,
    "01_data",
    "processed"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

revenue_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("=" * 60)
print("FINAL REVENUE DATASET SAVED")
print("=" * 60)

print("Output:", OUTPUT_PATH)
print("File exists:", os.path.exists(OUTPUT_PATH))
print("Rows saved:", len(revenue_df))
print("Columns saved:", len(revenue_df.columns))